In [1]:
from cropspectai import ml_config
from cropspectai.data_analysis import preprocessing
from cropspectai.modeling.train import train_model

import pandas as pd

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


# --------------------------------------------------
# Load data
# --------------------------------------------------

spectral_length = "as7343" # "full" or "1000" or "as7343"
target = "cab" # "cab" or None
indices = "all" # "all" or "established" or None

if spectral_length == "as7343":
    if target == None:
        train_path = (
            ml_config.PROCESSED_PARQUET_DATA_SPLITS_PATH
            / "all"
            / "train.parquet"
        )
    elif target == "cab":
        if indices == "all": # all engineered features
            train_path = (
                        ml_config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train_all_features.parquet"
                    )
        elif indices == "established": # only established new features
            train_path = (
                        ml_config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train_established_features.parquet"
                    ) 
        elif indices == None: # no engineered features
            train_path = (
                        ml_config.PROCESSED_PARQUET_DATA_SPLITS_PATH
                        / target
                        / "train.parquet"
                    )       
elif spectral_length == "full" or "1000":
    train_path = (
        ml_config.RAW_PARQUET_DATA_SPLITS_PATH
        / "train.parquet"
    )

df = pd.read_parquet(train_path)
print("Shape of imported data: ", df.shape)

Shape of imported data:  (1607, 41)


In [2]:
# --------------------------------------------------
# Get feature and target column names
# --------------------------------------------------

X, y = preprocessing.split_dataset_into_X_y(df, target)

targets = y.columns.tolist()

print("Targets:")
print(targets)

features = X.columns.tolist()
print("Features:")
print(features)

Targets:
['cab']
Features:
['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR', 'ci_red_edge_as7343', 'ci_green_as7343', 'gm_nd_as7343', 'red_edge_ratio_as7343', 'datt2_as7343', 'datt1_as7343', 'mcari_as7343', 'tcari_as7343', 'osavi_as7343', 'mcari_osavi_as7343', 'tcari_osavi_as7343', 'mndvi_as7343', 'ndvi_as7343', 'gndvi_as7343', 'sr_as7343', 'green_sr_as7343', 'evi2_as7343', 'rdvi_as7343', 'sipi_as7343', 'psri_as7343', 'red_green_nd', 'red_edge_green_nd', 'nir_green_nd', 'red_edge_slope_proxy', 'green_red_ratio', 'rededge_green_ratio', 'nir_fy_ratio', 'nir_fxl_ratio']


## X (features) as indices only

In [3]:
AS7343_CHANNELS = list(ml_config.AS7343_CHANNELS.keys())


X_index_features = X.drop(
        columns=AS7343_CHANNELS,
        errors="ignore",
    ).copy()

index_features = X_index_features.columns.tolist()

# print("Features:")
print(index_features)

# print(X.shape)

# df = pd.concat([X[AS7343_CHANNELS], y], axis = 1).copy()
# print(df.shape)


['ci_red_edge_as7343', 'ci_green_as7343', 'gm_nd_as7343', 'red_edge_ratio_as7343', 'datt2_as7343', 'datt1_as7343', 'mcari_as7343', 'tcari_as7343', 'osavi_as7343', 'mcari_osavi_as7343', 'tcari_osavi_as7343', 'mndvi_as7343', 'ndvi_as7343', 'gndvi_as7343', 'sr_as7343', 'green_sr_as7343', 'evi2_as7343', 'rdvi_as7343', 'sipi_as7343', 'psri_as7343', 'red_green_nd', 'red_edge_green_nd', 'nir_green_nd', 'red_edge_slope_proxy', 'green_red_ratio', 'rededge_green_ratio', 'nir_fy_ratio', 'nir_fxl_ratio']


## Hyperspectra up to 1000nm only

In [4]:
if spectral_length == "1000":
    print(X.shape)
    X_1000 = X.loc[:, "400":"1000"].copy()
    print(X_1000.shape)
    print(y.shape)

    df = pd.concat([X_1000, y], axis = 1).copy()
    print(df.shape)

In [5]:
# --------------------------------------------------
# Store results
# --------------------------------------------------

results = {}


# --------------------------------------------------
# Run XGBoost independently for every feature and every target
# --------------------------------------------------

for feature in index_features:

    print("\n" + "=" * 60)
    print(f"FEATURE: {feature}")
    print("=" * 60)

    X, y = preprocessing.split_dataset_into_X_y(df, target) # here "df" has multiple indices
    
    X_feature = pd.concat([X[AS7343_CHANNELS], X[[feature]]], axis = 1).copy()

    df_feature = pd.concat([X_feature, y], axis = 1).copy() # build new "df" with only one feature

    #check for duplicates
    df_feature = preprocessing.drop_duplicates(
        df_feature,
        target
    ).copy()

    # if df_feature changed, do this - here default
    X_feature, y = preprocessing.split_dataset_into_X_y(df_feature, target)

    X_feature = X_feature.copy()
    y = y.copy()

    # Train / validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X_feature,
        y,
        test_size=0.20,
        random_state=1,
    )

    # Create a fresh model for every target
    model_xgb = XGBRegressor(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=1.0,

        tree_method="hist",

        objective="reg:squarederror",
        eval_metric="rmse",

        n_jobs=-1,
        random_state=42,
    )

    # Train
    pipeline_xgb, metrics = train_model(
        model_xgb,
        X_train,
        y_train,
        X_val,
        y_val,
        scale=False,
    )

    metrics["Samples"] = len(df_feature)
    results[feature] = metrics


FEATURE: ci_red_edge_as7343
(Before) Shape of dataframe: (1607, 14)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 14)
Checking for rows without labels
Rows with no labels: 0 with target: cab
(After) Shape of dataframe: (1606, 14)

FEATURE: ci_green_as7343
(Before) Shape of dataframe: (1607, 14)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 14)
Checking for rows without labels
Rows with no labels: 0 with target: cab
(After) Shape of dataframe: (1606, 14)

FEATURE: gm_nd_as7343
(Before) Shape of dataframe: (1607, 14)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 14)
Checking for rows without labels
Rows with no labels: 0 with target: cab
(After) Shape of dataframe: (1606, 14)

FEATURE: red_edge_ratio_as7343
(Before) Shape of dataframe: (1607, 14)
Dropping duplicate rows
Duplicate rows: 1
(After) Shape of dataframe: (1606, 14)
Checking for rows without labels
Rows with no labels: 0 with target: c

In [6]:
# --------------------------------------------------
# Create results table
# --------------------------------------------------

results_df = pd.DataFrame(results)
print("\nXGBoost Results for each target")
print(results_df)


XGBoost Results for each target
                  ci_red_edge_as7343  ci_green_as7343  gm_nd_as7343  \
MAE (train)                 2.659728         2.534736      2.562684   
MAPE (train) [%]            8.472823         8.035495      8.201375   
MAE (test)                  7.399518         7.258011      7.368332   
MAPE (test) [%]            19.966149        19.818552     20.475969   
R² (test)                   0.477505         0.506202      0.496502   
Samples                  1606.000000      1606.000000   1606.000000   

                  red_edge_ratio_as7343  datt2_as7343  datt1_as7343  \
MAE (train)                    2.562684      2.559417      2.493041   
MAPE (train) [%]               8.201375      8.205160      7.896600   
MAE (test)                     7.368332      7.310178      7.175594   
MAPE (test) [%]               20.475969     20.270975     19.488350   
R² (test)                      0.496502      0.519502      0.522950   
Samples                     1606.000000   1

In [7]:
results_df.head()

,ci_red_edge_as7343,ci_green_as7343,gm_nd_as7343,red_edge_ratio_as7343,datt2_as7343,datt1_as7343,mcari_as7343,tcari_as7343,osavi_as7343,mcari_osavi_as7343,...,sipi_as7343,psri_as7343,red_green_nd,red_edge_green_nd,nir_green_nd,red_edge_slope_proxy,green_red_ratio,rededge_green_ratio,nir_fy_ratio,nir_fxl_ratio
MAE (train),2.659728,2.534736,2.562684,2.562684,2.559417,2.493041,2.607872,2.608144,2.594100,2.600470,...,2.528325,2.384495,2.556135,2.541216,2.534736,2.647588,2.568093,2.541216,2.546348,2.541712
MAPE (train) [%],8.472823,8.035495,8.201375,8.201375,8.205160,7.896600,8.324720,8.282147,8.322559,8.281832,...,8.108739,7.607996,8.070651,8.016438,8.035495,8.382667,8.103801,8.016438,8.056683,8.038816
MAE (test),7.399518,7.258011,7.368332,7.368332,7.310178,7.175594,7.387281,7.225441,7.428765,7.267210,...,7.262400,7.410449,7.070284,7.277204,7.258011,7.295874,7.057864,7.277204,7.303907,7.350759
MAPE (test) [%],19.966149,19.818552,20.475969,20.475969,20.270975,19.488350,20.290984,19.791171,20.419818,20.057918,...,20.026052,20.609516,19.312809,19.811553,19.818552,20.132402,19.267322,19.811553,20.072617,20.289536
R² (test),0.477505,0.506202,0.496502,0.496502,0.519502,0.522950,0.496621,0.506260,0.490700,0.503373,...,0.515714,0.493455,0.520837,0.506368,0.506202,0.503078,0.523490,0.506368,0.500396,0.494051


In [8]:
results_df.iloc[-2, :].sort_values(ascending=False)

tcari_osavi_as7343       0.529384
green_red_ratio          0.523490
datt1_as7343             0.522950
red_green_nd             0.520837
sr_as7343                0.519502
datt2_as7343             0.519502
ndvi_as7343              0.519502
sipi_as7343              0.515714
mndvi_as7343             0.515459
rededge_green_ratio      0.506368
red_edge_green_nd        0.506368
tcari_as7343             0.506260
gndvi_as7343             0.506202
ci_green_as7343          0.506202
green_sr_as7343          0.506202
nir_green_nd             0.506202
mcari_osavi_as7343       0.503373
red_edge_slope_proxy     0.503078
nir_fy_ratio             0.500396
rdvi_as7343              0.496727
mcari_as7343             0.496621
red_edge_ratio_as7343    0.496502
gm_nd_as7343             0.496502
nir_fxl_ratio            0.494051
psri_as7343              0.493455
evi2_as7343              0.490932
osavi_as7343             0.490700
ci_red_edge_as7343       0.477505
Name: R² (test), dtype: float64

In [9]:
top_4_indices = (
    results_df.iloc[-2, :]
    .sort_values(ascending=False)
    .head(4)
    .index
    .tolist()
)

print("top 4 indices in terms of R2 score: ", top_4_indices)

top 4 indices in terms of R2 score:  ['tcari_osavi_as7343', 'green_red_ratio', 'datt1_as7343', 'red_green_nd']


In [10]:
results_df.loc["MAPE (train) [%]", :].sort_values(ascending=True)

psri_as7343              7.607996
datt1_as7343             7.896600
rededge_green_ratio      8.016438
red_edge_green_nd        8.016438
gndvi_as7343             8.035495
ci_green_as7343          8.035495
nir_green_nd             8.035495
green_sr_as7343          8.035495
nir_fxl_ratio            8.038816
mndvi_as7343             8.049973
nir_fy_ratio             8.056683
red_green_nd             8.070651
green_red_ratio          8.103801
sipi_as7343              8.108739
gm_nd_as7343             8.201375
red_edge_ratio_as7343    8.201375
ndvi_as7343              8.205160
sr_as7343                8.205160
datt2_as7343             8.205160
mcari_osavi_as7343       8.281832
tcari_as7343             8.282147
evi2_as7343              8.310197
osavi_as7343             8.322559
mcari_as7343             8.324720
rdvi_as7343              8.343223
red_edge_slope_proxy     8.382667
ci_red_edge_as7343       8.472823
tcari_osavi_as7343       8.516940
Name: MAPE (train) [%], dtype: float64

In [11]:
results_df.loc["MAPE (test) [%]", :].sort_values(ascending=True)

green_red_ratio          19.267322
red_green_nd             19.312809
datt1_as7343             19.488350
tcari_as7343             19.791171
rededge_green_ratio      19.811553
red_edge_green_nd        19.811553
tcari_osavi_as7343       19.817576
gndvi_as7343             19.818552
ci_green_as7343          19.818552
nir_green_nd             19.818552
green_sr_as7343          19.818552
ci_red_edge_as7343       19.966149
sipi_as7343              20.026052
mcari_osavi_as7343       20.057918
nir_fy_ratio             20.072617
mndvi_as7343             20.091231
red_edge_slope_proxy     20.132402
ndvi_as7343              20.270975
sr_as7343                20.270975
datt2_as7343             20.270975
nir_fxl_ratio            20.289536
mcari_as7343             20.290984
rdvi_as7343              20.311247
osavi_as7343             20.419818
evi2_as7343              20.471604
red_edge_ratio_as7343    20.475969
gm_nd_as7343             20.475969
psri_as7343              20.609516
Name: MAPE (test) [%

In [12]:
# file_name = f"xgboost_results_{target}_{spectral_length}-{indices}-only.csv"
# output_path = ml_config.REPORTS_DIR / file_name

# results_df.to_csv(
#     output_path,
#     index=True,
# )

- full length hyperspectra gives better R2 score for all targets except CAB
- 1000nm cutoff hyperspectra gives consistently lower R2 score than full except for CAB
- as7343 does poor at all targets and consistently lower than 1000nm 
 - but CAB score remains on par with 1000nm
 - more importantly does better than even full

Conclusion:
- CAB is better predicted with data within 1000nm range
- as7343 is better capable at predicting CAB than all other targets